In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
from transformers import pipeline


PREGUNTAS_PERFIL = {
    "genero": {
        "texto": "¿Con qué género te identificas?",
        "opciones": {
            1: "Masculino",
            2: "Femenino",
            3: "No binario/Otro",
        },
    },
    "estilo": {
        "texto": "¿Qué estilo de ropa prefieres?",
        "opciones": {
            1: "Casual",
            2: "Formal",
            3: "Deportivo",
            4: "Elegante",
        },
    },
    "ocasion": {
        "texto": "¿Para qué ocasión buscas ropa?",
        "opciones": {
            1: "Uso diario",
            2: "Trabajo/Oficina",
            3: "Evento especial",
            4: "Deporte/Actividad física",
        },
    },
    "temporada": {
        "texto": "¿Para qué temporada buscas ropa?",
        "opciones": {
            1: "Primavera",
            2: "Verano",
            3: "Otoño",
            4: "Invierno",
        },
    },
    "presupuesto": {
        "texto": "¿Cuál es tu presupuesto?",
        "opciones": {
            1: "Bajo (económico)",
            2: "Medio",
            3: "Alto (premium)",
        },
    },
}


CATEGORIAS_POR_PERFIL = {
    "genero": {
        1: [0, 1, 2, 3, 4, 6],
        2: [0, 1, 2, 3, 4, 7],
        3: [0, 1, 2, 4, 6, 7],
    },
    "estilo": {
        1: [0, 1, 2, 5, 7, 9],
        2: [3, 4, 6, 8],
        3: [0, 1, 5, 9],
        4: [3, 4, 6, 7, 8],
    },
    "ocasion": {
        1: [0, 1, 2, 5, 9],
        2: [1, 4, 6, 8],
        3: [3, 4, 6, 7, 8],
        4: [0, 1, 5, 9],
    },
    "temporada": {
        1: [0, 1, 6, 7],
        2: [0, 1, 3, 5, 7],
        3: [2, 6, 7, 8],
        4: [2, 4, 9],
    },
}

PESOS_PERFIL = {
    "genero": 1.0,
    "estilo": 2.0,
    "ocasion": 2.0,
    "temporada": 1.5,
}

MODELO_CLASIFICADOR_TEXTO = None


def cargar_dataset_fashion_mnist():
    datos, metadatos = tfds.load("fashion_mnist", as_supervised=True, with_info=True)
    datos_entrenamiento, datos_pruebas = datos["train"], datos["test"]
    nombres_clases = metadatos.features["label"].names

    def normalizar(imagen, etiqueta):
        imagen = tf.cast(imagen, tf.float32) / 255.0
        return imagen, etiqueta

    datos_entrenamiento = (
        datos_entrenamiento
        .map(normalizar, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .shuffle(1000)
        .batch(32)
        .prefetch(tf.data.AUTOTUNE)
    )

    datos_pruebas = (
        datos_pruebas
        .map(normalizar, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .batch(32)
        .prefetch(tf.data.AUTOTUNE)
    )

    return datos_entrenamiento, datos_pruebas, nombres_clases


def crear_modelo_cnn():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])

    modelo.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return modelo


def cargar_modelo_base(epochs=5):
    datos_entrenamiento, datos_pruebas, nombres_clases = cargar_dataset_fashion_mnist()
    modelo = crear_modelo_cnn()

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True,
    )

    historial = modelo.fit(
        datos_entrenamiento,
        epochs=epochs,
        validation_data=datos_pruebas,
        callbacks=[early_stopping],
        verbose=1,
    )

    loss_prueba, acc_prueba = modelo.evaluate(datos_pruebas, verbose=0)
    print(f"\nRendimiento en test -> loss: {loss_prueba:.4f}, accuracy: {acc_prueba:.2%}")

    return modelo, nombres_clases, datos_entrenamiento, historial


def solicitar_opcion(pregunta, opciones):
    print(f"\n{pregunta}")
    for clave, etiqueta in opciones.items():
        print(f"{clave}. {etiqueta}")

    while True:
        valor = input(f"Selecciona una opción ({min(opciones)}-{max(opciones)}): ").strip()
        if not valor.isdigit():
            print("Entrada no válida. Debes escribir un número.")
            continue

        opcion = int(valor)
        if opcion in opciones:
            return opcion

        print("Opción fuera de rango. Intenta de nuevo.")


def sistema_perfilado_usuario():
    perfil = {}
    for campo, spec in PREGUNTAS_PERFIL.items():
        perfil[campo] = solicitar_opcion(spec["texto"], spec["opciones"])
    return perfil


def generar_recomendaciones(perfil, nombres_clases, top_k=5):
    puntajes_por_categoria = {i: 0.0 for i in range(len(nombres_clases))}

    for campo in ["genero", "estilo", "ocasion", "temporada"]:
        categorias = CATEGORIAS_POR_PERFIL[campo][perfil[campo]]
        peso = PESOS_PERFIL[campo]
        for categoria in categorias:
            puntajes_por_categoria[categoria] += peso

    categorias_ordenadas = sorted(
        puntajes_por_categoria.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    categorias_filtradas = [
        categoria for categoria, score in categorias_ordenadas if score > 0
    ][:top_k]

    nivel_precio = ["económico", "precio medio", "premium"][perfil["presupuesto"] - 1]

    recomendaciones = [
        f"{nombres_clases[categoria].title()} ({nivel_precio})"
        for categoria in categorias_filtradas
    ]

    return recomendaciones, categorias_filtradas


def mostrar_ejemplos_visuales(indices_clases, nombres_clases, datos_entrenamiento, max_por_categoria=3):
    if not indices_clases:
        print("No hay categorías para mostrar.")
        return

    ejemplos_por_categoria = {indice: [] for indice in indices_clases}

    for imagen, etiqueta in datos_entrenamiento.unbatch():
        etiqueta = int(etiqueta.numpy())
        if etiqueta in ejemplos_por_categoria and len(ejemplos_por_categoria[etiqueta]) < max_por_categoria:
            ejemplos_por_categoria[etiqueta].append(imagen.numpy().reshape((28, 28)))

        if all(len(imgs) >= max_por_categoria for imgs in ejemplos_por_categoria.values()):
            break

    total = sum(len(imgs) for imgs in ejemplos_por_categoria.values())
    if total == 0:
        print("No se encontraron ejemplos visuales para las recomendaciones.")
        return

    columnas = 3
    filas = (total + columnas - 1) // columnas
    plt.figure(figsize=(12, filas * 3))

    idx = 1
    for clase_idx in indices_clases:
        for img in ejemplos_por_categoria[clase_idx]:
            plt.subplot(filas, columnas, idx)
            plt.xticks([])
            plt.yticks([])
            plt.imshow(img, cmap=plt.cm.binary)
            plt.xlabel(nombres_clases[clase_idx].title())
            idx += 1

    plt.tight_layout()
    plt.show()


def obtener_clasificador_texto():
    global MODELO_CLASIFICADOR_TEXTO
    if MODELO_CLASIFICADOR_TEXTO is None:
        MODELO_CLASIFICADOR_TEXTO = pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
        )
    return MODELO_CLASIFICADOR_TEXTO


def recomendar_por_texto(descripcion_usuario, nombres_clases, top_k=5):
    if not descripcion_usuario.strip():
        return []

    clasificador = obtener_clasificador_texto()
    etiquetas = [c.title() for c in nombres_clases]
    resultado = clasificador(descripcion_usuario, etiquetas)

    recomendaciones_nlp = list(zip(resultado["labels"][:top_k], resultado["scores"][:top_k]))

    print("\nRecomendaciones basadas en tu descripción libre:")
    for etiqueta, score in recomendaciones_nlp:
        print(f"- {etiqueta} ({score:.2%})")

    return recomendaciones_nlp


def unificar_recomendaciones(recomendaciones_perfil, recomendaciones_nlp):
    ranking = {}

    for idx, recomendacion in enumerate(recomendaciones_perfil):
        etiqueta = recomendacion.split(" (")[0].strip().title()
        ranking[etiqueta] = ranking.get(etiqueta, 0.0) + (1.0 / (idx + 1))

    for etiqueta, score in recomendaciones_nlp:
        etiqueta_limpia = etiqueta.strip().title()
        ranking[etiqueta_limpia] = ranking.get(etiqueta_limpia, 0.0) + score

    recomendaciones_finales = sorted(ranking.items(), key=lambda item: item[1], reverse=True)
    return recomendaciones_finales


def main():
    print("Bienvenido al Sistema de Recomendación de Prendas de Vestir")

    modelo, nombres_clases, datos_entrenamiento, historial = cargar_modelo_base(epochs=5)
    _ = modelo, historial

    perfil = sistema_perfilado_usuario()
    recomendaciones_perfil, indices_clases = generar_recomendaciones(perfil, nombres_clases, top_k=5)

    print("\nBasado en tus preferencias, te recomendamos:")
    for i, rec in enumerate(recomendaciones_perfil, 1):
        print(f"{i}. {rec}")

    descripcion_usuario = input("\nEscribe libremente cómo te gustaría tu outfit o prenda ideal: ")
    recomendaciones_nlp = recomendar_por_texto(descripcion_usuario, nombres_clases, top_k=5)

    recomendaciones_finales = unificar_recomendaciones(recomendaciones_perfil, recomendaciones_nlp)

    print("\nLista final unificada de recomendaciones:")
    for etiqueta, score in recomendaciones_finales:
        print(f"- {etiqueta} (score combinado: {score:.3f})")

    print("\nMostrando ejemplos visuales de las prendas recomendadas...")
    mostrar_ejemplos_visuales(indices_clases, nombres_clases, datos_entrenamiento)


if __name__ == "__main__":
    main()

SyntaxError: invalid non-printable character U+00A0 (ipython-input-3135977672.py, line 11)